In [ ]:
from sodapy import Socrata
import pandas as pd
from google.oauth2 import service_account
from googleapiclient.discovery import build
import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import os
bic_etl_home = os.getenv('bic_etl_home')

In [ ]:
def getDatasetTrackerInfo(bic_etl_home):

    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
                 "https://www.googleapis.com/auth/drive.file",
                      "https://www.googleapis.com/auth/drive"]
    
    #creds = ServiceAccountCredentials.from_json_keyfile_name('../../scripts/client_secret.json',
    #    scope)
    creds = ServiceAccountCredentials.from_json_keyfile_name(os.path.join(bic_etl_home, 'general', 'scripts','client_secret.json'),scope)
    
    client = gspread.authorize(creds)
    tracker = client.open('BIC Dataset Tracker').worksheet(
    'PublishedData')
    dfTracker = pd.DataFrame(tracker.get_all_records(head=2))
    return dfTracker

def getCimDatasets():
    cim_url_query = "data.colorado.gov"
    allDatasets = []
    cimDatasets = {}
    
    # Connect to the Socrata API
    with Socrata(cim_url_query, None) as client:
        datasets = client.datasets()
        for dataset in datasets:
            allDatasets.append(dataset)
            if dataset['owner']['display_name'] == 'Colorado Information Marketplace' or dataset['owner']['display_name'] == "Business Intelligence Center of CO":
                title=dataset["resource"]["name"]
                w4x4=dataset["resource"]["id"]

                cimDatasets[w4x4]=dataset
    
    return cimDatasets
